In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Credit Card Fraud Detection — Data Preprocessing

## Project: CardGuard

### Objective

The objective of this stage is to transform the raw credit-card transaction data into a clean, validated, and machine-learning-ready representation.

The preprocessing pipeline is designed to:

- Preserve data integrity
- Prevent data leakage
- Handle missing and invalid values
- Convert variables into appropriate data types
- Remove non-predictive identifiers
- Prepare numerical features
- Encode categorical variables
- Maintain consistency between training and inference
- Support reproducible model development

All transformations that learn parameters from the data must be fitted using training data only.

In [19]:
data = pd.read_csv(r'D:\Creditcard_fraud_detection\data\raw\fraudTrainR.csv')
# data = pd.read_csv(r'../data/raw/fraudTest.csv')
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)
data.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,5,2019-01-01 00:04:08,4767265376804500,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,Dublin,PA,18917,40.3750,-75.2045,2158,Transport planner,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653382,-76.152667,0
1,7,2019-01-01 00:05:08,6011360759745864,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,Edinburg,VA,22824,38.8432,-78.6003,6018,"Designer, multimedia",1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948089,-78.540296,0
2,13,2019-01-01 00:07:27,5559857416065248,fraud_Kiehn Inc,grocery_pos,96.29,Jack,Hill,M,5916 Susan Bridge Apt. 939,Grenada,CA,96038,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,1325376447,41.657520,-122.230347,0
3,14,2019-01-01 00:09:03,3514865930894695,fraud_Beier-Hyatt,shopping_pos,7.77,Christopher,Castaneda,M,1632 Cohen Drive Suite 639,High Rolls Mountain Park,NM,88325,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,1325376543,32.863258,-106.520205,0
4,23,2019-01-01 00:17:40,630441765090,fraud_Pacocha-Bauch,shopping_pos,9.55,Susan,Washington,F,759 Erin Mount Suite 956,May,TX,76857,31.9571,-98.9656,1791,Corporate investment banker,1965-07-26,c4b4daebab8be54cadde4b941244ca53,1325377060,31.626350,-98.610225,0


In [20]:
df = data.copy()

print(df.shape)

(222873, 23)


Schema validation

In production, the pipeline should fail if the incoming data suddenly changes.

In [21]:
expected_columns = [
    "trans_date_trans_time",
    "cc_num",
    "merchant",
    "category",
    "amt",
    "first",
    "last",
    "gender",
    "street",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "unix_time",
    "merch_lat",
    "merch_long",
    "is_fraud"
]

missing_columns = [col for col in expected_columns
    if col not in df.columns]

extra_columns = [col for col in df.columns
    if col not in expected_columns]

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

Missing columns: []
Extra columns: ['Unnamed: 0', 'city', 'state', 'zip']


In [22]:
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

In [23]:
print(df["is_fraud"].value_counts(dropna=False))

is_fraud
0    215367
1      7506
Name: count, dtype: int64


In [24]:
invalid_target = ~df["is_fraud"].isin([0, 1])

print("Invalid target rows:",invalid_target.sum())

Invalid target rows: 0


In [25]:
if invalid_target.any():
    raise ValueError("Target contains values other than 0 and 1.")

Convert datetime columns

In [26]:
datetime_columns = ["trans_date_trans_time","dob"]

for col in datetime_columns:
    df[col] = pd.to_datetime(df[col],errors="coerce")

Validate datetime conversion

In [27]:
for col in datetime_columns:
    invalid_dates = df[col].isna().sum()
    print(f"{col}: {invalid_dates:,} invalid/missing dates")

trans_date_trans_time: 0 invalid/missing dates
dob: 0 invalid/missing dates


Check missing values

In [28]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage":(df.isna().mean() * 100)
})

missing_report.sort_values("missing_percentage",ascending=False)

,missing_count,missing_percentage
Unnamed: 0,0,0.0
trans_date_trans_time,0,0.0
cc_num,0,0.0
merchant,0,0.0
category,0,0.0
amt,0,0.0
first,0,0.0
last,0,0.0
gender,0,0.0
street,0,0.0


Handle missing values

In [29]:
duplicate_mask = df.duplicated()

print(f"Duplicate rows: {duplicate_mask.sum():,}")

Duplicate rows: 0


Validate numerical ranges

A machine-learning pipeline should detect impossible values.

In [30]:
range_checks = {
    "lat": (-90, 90),
    "long": (-180, 180),
    "merch_lat": (-90, 90),
    "merch_long": (-180, 180),
    "city_pop": (0, np.inf),
    "amt": (0, np.inf)
}

for col, (lower, upper) in range_checks.items():
    invalid = ((df[col] < lower) |(df[col] > upper)).sum()

    print(f"{col}: {invalid:,} invalid values")

lat: 0 invalid values
long: 0 invalid values
merch_lat: 0 invalid values
merch_long: 0 invalid values
city_pop: 0 invalid values
amt: 0 invalid values


Identify identifier columns

In [31]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

Handle customer names/address

In [32]:
df.columns

Index(['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud'],
      dtype='str')

final preprocessing validation report

In [33]:
print("=" * 60)
print("FINAL DATA PREPROCESSING VALIDATION")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nInfinite values:")
print(
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

print("\nTarget distribution:")
print(df["is_fraud"].value_counts())

print("\nData types:")
print(df.dtypes)

FINAL DATA PREPROCESSING VALIDATION

Shape:
(222873, 22)

Missing values:
0

Duplicate rows:
0

Infinite values:
0

Target distribution:
is_fraud
0    215367
1      7506
Name: count, dtype: int64

Data types:
trans_date_trans_time    datetime64[us]
cc_num                            int64
merchant                            str
category                            str
amt                             float64
first                               str
last                                str
gender                              str
street                              str
city                                str
state                               str
zip                               int64
lat                             float64
long                            float64
city_pop                          int64
job                                 str
dob                      datetime64[us]
trans_num                           str
unix_time                         int64
merch_lat                      

In [34]:
df.to_csv("../data/processed/creditcard_preprocessed.csv",index=False)

print("Preprocessed data saved successfully.")
print("Shape:", df.shape)

Preprocessed data saved successfully.
Shape: (222873, 22)
